# N-Gram Analysis
This notebook aims at investigating the most frequent n-grams in music-related YouTube videos.

In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import json

with open(os.path.join("data", "dvi2", "dataset", "train.json"), "r") as f:
    train_data = json.load(f)
    
print("Loading DVI2 dataset...")
dvi2_path = os.path.join("data", "dvi2", "dataset", "matched", "dvi_fm_filtered.jsonl")
df_dvi2 = pd.read_json(dvi2_path, lines=True, orient="records")
# Explode versions
df_exp = df_dvi2.explode("versions")
df_exp = pd.concat(
    [df_exp[["clique_id"]].reset_index(drop=True),
     pd.json_normalize(df_exp["versions"])],
    axis=1
)
df_exp = df_exp.explode("tracks").reset_index(drop=True)
df_exp = pd.concat(
    [df_exp[["clique_id", "version_id"]],
     pd.json_normalize(df_exp["tracks"])],
    axis=1
)
df_exp = df_exp.explode("youtube_video").reset_index(drop=True)
df_dvi2 = pd.concat(
    [df_exp,
     pd.json_normalize(df_exp["youtube_video"])],
    axis=1
)
df_dvi2 = df_dvi2.drop_duplicates(subset=["clique_id", "version_id", "url"])
def get_youtube_id(row):
    if pd.isna(row["url"]):
        return row["version_id"]
    else:
        return row["url"].split("watch?v=")[-1]
df_dvi2["youtube_id"] = df_dvi2.apply(get_youtube_id, axis=1)

print("Loading YT data...")
df_yt = pd.read_parquet(os.path.join("data", "yt.parquet")).reset_index(drop=True)

df = pd.merge(df_dvi2, df_yt, on="youtube_id", how="left")
df = df.dropna(subset=["title", "description"])


Loading DVI2 dataset...


### Clique stopwords (title, artist, writers)

In [ ]:
def concat_fields(row):
    title, writers, artists = row.track_title, row.track_writer_names, row.track_artist_names
    return [title] + (writers if isinstance(writers, list) else [""]) + (artists if isinstance(artists, list) else [""])
    
df_dvi2.loc[:,"clique_stopwords"] = df_dvi2.apply(concat_fields, axis=1)
df_dvi2.loc[:,"clique_stopwords"] = df_dvi2.clique_stopwords.apply(lambda x: [s.lower() for s in x if isinstance(s, str)])
df_stopwords = df_dvi2[["clique_id", "clique_stopwords"]].groupby("clique_id").agg(
    {"clique_stopwords": lambda x: list(set([item for sublist in x for item in sublist]))}).reset_index()


In [89]:
from rapidfuzz import fuzz

def mask_stopwords(text, stopwords, threshold=90):
    if not isinstance(text, str) or not stopwords:
        return text
    tokens = text.split()
    masked = []
    for token in tokens:
        found = False
        for sw in stopwords:
            if sw and fuzz.token_sort_ratio(token.lower(), sw.lower()) >= threshold:
                found = True
                break
        masked.append("[MASK]" if found else token)
    return " ".join(masked)

def mask_stopwords_list(tags, stopwords, threshold=90):
    if not isinstance(tags, list) or not stopwords:
        return tags
    masked = []
    for tag in tags:
        found = False
        for sw in stopwords:
            if sw and fuzz.token_sort_ratio(str(tag).lower(), sw.lower()) >= threshold:
                found = True
                break
        masked.append("[MASK]" if found else tag)
    return masked

# Prepare stopword mapping
stopword_map = dict(zip(df_stopwords.clique_id, df_stopwords.clique_stopwords))

tqdm.pandas(desc="Masking stopwords in titles, descriptions, and tags")
df["title2"] = df.progress_apply(lambda row: mask_stopwords(row["title"], stopword_map.get(row["clique_id"], [])), axis=1)
df["description2"] = df.progress_apply(lambda row: mask_stopwords(row["description"], stopword_map.get(row["clique_id"], [])), axis=1)
df["tags2"] = df.progress_apply(lambda row: mask_stopwords_list(row["tags"] if "tags" in row else [], stopword_map.get(row["clique_id"], [])), axis=1)


Masking stopwords in titles, descriptions, and tags: 100%|██████████| 2077862/2077862 [00:33<00:00, 61183.52it/s]
Masking stopwords in titles, descriptions, and tags: 100%|██████████| 2077862/2077862 [03:07<00:00, 11060.72it/s]
Masking stopwords in titles, descriptions, and tags: 100%|██████████| 2077862/2077862 [00:17<00:00, 117459.78it/s]


### Stemming + Extraction

In [ ]:
from nltk.util import ngrams
from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

from preprocessing.string_processor import StringProcessor

processor = StringProcessor()

stemmer = SnowballStemmer("english")  # you can switch based on language if needed

def tokenize_and_stem(text, stem=True, processor=lambda x: x):
    if not isinstance(text, str):
        return []
    text = processor(text)  # Preprocess
    tokens = word_tokenize(text.lower())
    if stem:
        tokens = [stemmer.stem(t) for t in tokens]
    return tokens

def extract_ngrams(tokens, n):
    return ['_'.join(ng) for ng in ngrams(tokens, n)]

def get_all_ngrams(text, stem=True, processor=lambda x: x):
    tokens = tokenize_and_stem(text, stem=stem, processor=processor)
    return {
        '1grams': extract_ngrams(tokens, 1),
        '2grams': extract_ngrams(tokens, 2),
        '3grams': extract_ngrams(tokens, 3)
    }


def process_column(df, colname, stem=True, processor=lambda x: x):
    df.loc[:,f'{colname}_1grams'] = df[colname].progress_apply(lambda x: get_all_ngrams(x, stem, processor)['1grams'])
    df.loc[:,f'{colname}_2grams'] = df[colname].progress_apply(lambda x: get_all_ngrams(x, stem, processor)['2grams'])
    df.loc[:,f'{colname}_3grams'] = df[colname].progress_apply(lambda x: get_all_ngrams(x, stem, processor)['3grams'])

def process_tags_column(df, colname, stem=True, processor=lambda x: x):
    def process_tag_list(taglist):
        if not isinstance(taglist, list):
            return [], [], []
        # Apply processor + stemming to each tag
        tokens = []
        for tag in taglist:
            tag = processor(tag)
            tokens.extend(tokenize_and_stem(tag, stem=stem, processor=lambda x: x))  # processor already applied
        return (
            extract_ngrams(tokens, 1),
            extract_ngrams(tokens, 2),
            extract_ngrams(tokens, 3)
        )

    results = df[colname].progress_apply(process_tag_list)
    df.loc[:,f'{colname}_1grams'] = results.progress_apply(lambda x: x[0])
    df.loc[:,f'{colname}_2grams'] = results.progress_apply(lambda x: x[1])
    df.loc[:,f'{colname}_3grams'] = results.progress_apply(lambda x: x[2])

process_column(df, "title2", stem=True, processor=processor)
process_column(df, "description2", stem=True, processor=processor)
process_tags_column(df, "tags2", stem=True, processor=processor)


[nltk_data] Downloading package punkt to /home/sha/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Translating titles, descriptions, and tags to English: 100%|██████████| 1330494/1330494 [02:11<00:00, 10117.24it/s]
/tmp/ipykernel_343672/1473842295.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'{colname}_1grams'] = df[colname].progress_apply(lambda x: get_all_ngrams(x, stem, processor)['1grams'])
Translating titles, descriptions, and tags to English: 100%|██████████| 1330494/1330494 [02:11<00:00, 10126.95it/s]
/tmp/ipykernel_343672/1473842295.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the do

### Most frequent N-Grams

### Translate to English

In [108]:
from easynmt import EasyNMT

model = EasyNMT('opus-mt')

def translate_text(text, target_lang="en"):
    try:
        if isinstance(text, str):
            return model.translate(text, target_lang=target_lang)
        else:
            return text
    except Exception as e:
        print(f"Translation error: {e}")
        return text

tqdm.pandas(desc="Translating titles, descriptions, and tags to English")
df["title2_en"] = df.title2.progress_apply(lambda x: translate_text(x))
df["description2_en"] = df.description2.progress_apply(lambda x: translate_text(x))
df["tags2_en"] = df.tags2.progress_apply(lambda x: [model.translate_text(tag) for tag in x] if isinstance(x, list) else [])


Translating titles, descriptions, and tags to English:   0%|          | 42/1330494 [00:00<1:47:19, 206.60it/s]Exception: Helsinki-NLP/opus-mt-sw-en is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`
Exception: Helsinki-NLP/opus-mt-sw-en is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`
Translating titles, descriptions, and tags to English:   0%|          | 63/1330494 [00:03<23:45:12, 15.56it/s]

Translation error: Helsinki-NLP/opus-mt-sw-en is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`
Translation error: Helsinki-NLP/opus-mt-sw-en is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`


Translating titles, descriptions, and tags to English:   0%|          | 167/1330494 [00:18<41:12:57,  8.97it/s]


KeyboardInterrupt: 

In [103]:
title2_en = model.translate(df.title2.fillna("").to_list(), target_lang="en")


Exception: No method for automatic language detection was found. Please install at least one of the following: fasttext (pip install fasttext), langid (pip install langid), or langdetect (pip install langdetect)

### Get general 

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

from collections import Counter
from itertools import islice
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import json

# Set up stopwords
languages = [
    'arabic', 'azerbaijani', 'danish', 'dutch', 'english', 'finnish',
    'french', 'german', 'greek', 'hebrew', 'hungarian', 'indonesian',
    'italian', 'kazakh', 'nepali', 'norwegian', 'portuguese', 'romanian',
    'russian', 'slovene', 'spanish', 'swedish', 'tajik', 'turkish'
]
stop_words = set()
for lang in languages:
    stop_words.update(stopwords.words(lang))

def ngrams(tokens, n):
    return zip(*(islice(tokens, i, None) for i in range(n)))

def clean_and_tokenize(text):
    tokens = word_tokenize(text.lower())
    return [word for word in tokens if word.isalnum() and word not in stop_words]

def get_top_ngrams(series, n=1, top_k=500):
    all_ngrams = Counter()
    for text in series.dropna().astype(str):
        tokens = clean_and_tokenize(text)
        all_ngrams.update(ngrams(tokens, n))
    return {" ".join(gram): count for gram, count in all_ngrams.most_common(top_k)}

# Final result dictionary
final_ngrams = {}

# Assuming your DataFrame is called metadata
top_k = 500  # Set the number of top n-grams to retrieve
for column in ['title', 'description', 'tags_str']:
    final_ngrams[column] = {}
    for n in [1, 2, 3]:
        top_ngrams = get_top_ngrams(metadata[column], n=n, top_k=top_k)
        final_ngrams[column][str(n)] = top_ngrams

# Save as clean, structured JSON
with open(f"data/analysis/top_{top_k}_ngrams.json", "w", encoding="utf-8") as f:
    json.dump(final_ngrams, f, ensure_ascii=False, indent=2)


In [ ]:
import nltk
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from itertools import islice

# Ensure required NLTK resources are downloaded
nltk.download('punkt')
nltk.download('stopwords')

def analyze_top_ngrams(
    df,
    ngram_range=(1, 3),
    top_k=15,
    languages=('english', 'german', 'french', 'spanish', 'italian', 'portuguese')
):
    """
    Analyzes top n-grams in specified text columns of a DataFrame after cleaning and stopword removal.

    Args:
        df (pd.DataFrame): DataFrame containing text data.
        ngram_range (tuple): (min_n, max_n) tuple for n-gram lengths.
        top_k (int): Number of top n-grams to return.
        languages (tuple): Languages whose stopwords to combine.

    Returns:
        dict: Nested dictionary of top n-grams per column and n.
    """
    # Compile stopwords from multiple languages
    stop_words = set()
    for lang in languages:
        stop_words.update(stopwords.words(lang))

    def ngrams(tokens, n):
        return zip(*(islice(tokens, i, None) for i in range(n)))

    def clean_and_tokenize(text):
        tokens = word_tokenize(text.lower())
        return [word for word in tokens if word.isalnum() and word not in stop_words]

    results = {}

    text_columns = ['title', 'description', 'tags_str']
    for column in text_columns:
        print(f"\n==== {column.upper()} ====")
        col_results = {}
        for n in range(ngram_range[0], ngram_range[1] + 1):
            all_ngrams = Counter()
            for text in df[column].dropna().astype(str):
                tokens = clean_and_tokenize(text)
                all_ngrams.update(ngrams(tokens, n))
            top = all_ngrams.most_common(top_k)

            print(f"\nTop {n}-grams (stopwords removed):")
            for phrase, count in top:
                joined = " ".join(phrase) if isinstance(phrase, tuple) else phrase
                print(f"{joined}: {count}")

            col_results[n] = top
        results[column] = col_results

    return results

results_metadata = analyze_top_ngrams(
    metadata,
    ngram_range=(1, 3),
    top_k=15
)

